# Keyword Co-occurrence Graph

Builds an undirected article similarity graph from keyword co-occurrence and co-authorship signals. Articles sharing enough high-strength keywords are connected by weighted edges; the graph is then pruned with a mutual top-K neighbour filter to keep only the strongest links.

This graph is the foundation for the Leiden clustering in `cluster_graph_vis.ipynb`.

**Input:** `output/metadata_enriched_FINAL.csv`, `keywords_results/keywords_normalized_deduplicated_no_stopwords.csv`  
**Output:** `output/articles_no_clusters.graphml`, `output/articles_no_clusters.html`

In [9]:
import ast
import itertools
from collections import defaultdict
from pathlib import Path

import pandas as pd
import networkx as nx
from pyvis.network import Network

OUTPUT_DIR = Path("..").resolve() / "output"
KW_DIR     = Path("..").resolve() / "keywords_results"

META_PATH   = OUTPUT_DIR / "metadata_enriched_FINAL.csv"
KW_PATH     = KW_DIR    / "keywords_normalized_deduplicated_no_stopwords.csv"

GRAPHML_OUT = OUTPUT_DIR / "articles_no_clusters.graphml"
HTML_OUT    = OUTPUT_DIR / "articles_no_clusters.html"

**Load data.** Reads the enriched article metadata and the normalised keyword CSV into DataFrames.

In [10]:
meta = pd.read_csv(META_PATH)
kw = pd.read_csv(KW_PATH)

print("meta shape:", meta.shape)
print("kw shape:", kw.shape)
print(meta.columns.tolist())
print(kw.columns.tolist())

meta shape: (2376, 22)
kw shape: (2366, 2)
['pdf_file', 'original_title', 'original_authors', 'original_year', 'enriched', 'doi', 'crossref_title', 'crossref_authors', 'crossref_year', 'title_similarity', 'crossref_references', 'crossref_subject', 'crossref_journal', 'crossref_publisher', 'crossref_type', 'crossref_abstract', 'crossref_issn', 'crossref_volume', 'crossref_issue', 'crossref_page', 'crossref_event_name', 'crossref_language']
['filename', 'keywords_without_stopwords']


**Helper functions.** Utilities for ID normalisation, author splitting, title selection, and keyword strength parsing.

In [11]:
def normalize_id_from_pdf(pdf_name: str) -> str:
    """Return a normalised lowercase article ID derived from a PDF filename."""
    if pd.isna(pdf_name):
        return ""
    s = str(pdf_name).strip()
    if s.lower().endswith(".pdf"):
        s = s[:-4]
    return s.lower()

def normalize_id_from_filename(filename: str) -> str:
    """Return a normalised lowercase article ID derived from a keyword CSV filename."""
    if pd.isna(filename):
        return ""
    return str(filename).strip().lower()

def split_authors(raw_authors: str) -> list:
    """Split a semicolon-delimited author string into a list of stripped name strings."""
    if pd.isna(raw_authors):
        return []
    # metadata uses ; as separator in original_authors/crossref_authors
    parts = [a.strip() for a in str(raw_authors).split(";")]
    return [p for p in parts if p]

def pick_title(row) -> str:
    """Return the best available title: Crossref first, then original, then pdf_file."""
    t = row.get("crossref_title")
    if isinstance(t, str) and t.strip():
        return t.strip()
    t = row.get("original_title")
    if isinstance(t, str) and t.strip():
        return t.strip()
    return row.get("pdf_file", "unknown_title")

def pick_authors(row) -> list:
    """Return authors as a list, preferring Crossref over original metadata."""
    a = row.get("crossref_authors")
    if isinstance(a, str) and a.strip():
        return split_authors(a)
    return split_authors(row.get("original_authors"))

def parse_keywords_with_strength(cell):
    """
    Expected format in file:
    "[('keyword one', 3.2), ('keyword two', 2.5), ...]"
    Returns list[(keyword, strength_float)].
    """
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
    except Exception:
        return []
    
    out = []
    for item in parsed:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            kw = str(item[0]).strip().lower()
            try:
                strength = float(item[1])
            except Exception:
                continue
            if kw:
                out.append((kw, strength))
    return out

**Build article table.** Merges metadata and keywords on article ID, producing one row per article with title, authors, and a list of `(keyword, strength)` pairs.

In [12]:
meta = meta.copy()
kw = kw.copy()

meta["article_id"] = meta["pdf_file"].map(normalize_id_from_pdf)
kw["article_id"] = kw["filename"].map(normalize_id_from_filename)

# Keep only needed metadata columns
meta_small = meta[["article_id", "pdf_file", "original_title", "crossref_title", "original_authors", "crossref_authors"]].copy()

# Parse keyword list with strengths
kw["keyword_items"] = kw["keywords_without_stopwords"].map(parse_keywords_with_strength)

# Merge. Left join from metadata to keep all article nodes.
articles = meta_small.merge(
    kw[["article_id", "keyword_items"]],
    on="article_id",
    how="left"
)

# Fill missing keywords with empty list
articles["keyword_items"] = articles["keyword_items"].apply(lambda x: x if isinstance(x, list) else [])

# Build final per-article fields
articles["title"] = articles.apply(pick_title, axis=1)
articles["authors"] = articles.apply(pick_authors, axis=1)

print("articles:", len(articles))
articles.head(3)

articles: 2376


,article_id,pdf_file,original_title,crossref_title,original_authors,crossref_authors,keyword_items,title,authors
0,00alissandrakisnehanivdeutenhan,00AlissandrakisNehanivDeutenhan.pdf,Learning How to Do Things with Imitation,NaN,Aris Alissandrakis; Chrystopher Nehaniv; Kerst...,NaN,"[(imitation learning, 4.833333333333333), (che...",Learning How to Do Things with Imitation,"[Aris Alissandrakis, Chrystopher Nehaniv, Kers..."
1,00baenabelmontemandow,00BaenaBelmonteMandow.pdf,An Intelligent Tutor for a Web-Based Chess Course,An Intelligent Tutor for a Web-Based Chess Course,Antonio Baena; María-Victoria Belmonte; Lawren...,Antonio Baena; María-Victoria Belmonte; Lawren...,"[(intelligent tutoring system, 5.0), (adaptive...",An Intelligent Tutor for a Web-Based Chess Course,"[Antonio Baena, María-Victoria Belmonte, Lawre..."
2,00bailey,00Bailey.pdf,Figure 8,Gabaergic control of odour-induced activity in...,Marcel Duchamp; Jacques Villon,P. Duchamp-Viret; A. Duchamp,"[(chess moralities, 3.0), (chess terminology, ...",Gabaergic control of odour-induced activity in...,"[P. Duchamp-Viret, A. Duchamp]"


**Build keyword co-occurrence graph.** For every pair of articles that share keywords, computes an aggregated edge score, then prunes to the top-K strongest neighbours per node using mutual kNN.

In [13]:
# Tuning parameters
MIN_SHARED_KEYWORDS = 3
MIN_EDGE_STRENGTH = 3.2
MIN_MAX_PAIR_SCORE = 3.5
TOP_K_PER_NODE = 8
USE_MUTUAL_KNN = True
AUTHOR_BOOST = 0.25
INCLUDE_AUTHOR_ONLY_EDGES = False
MAX_TOTAL_EDGES = None  # set a number only as emergency safety cap

G = nx.Graph()

# Add article nodes
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    title = row["title"]
    authors = row["authors"]
    kw_items = row["keyword_items"]

    G.add_node(
        article_id,
        label=title,
        title=title,
        pdf_file=str(row["pdf_file"]),
        authors="|".join(authors),
        n_authors=len(authors),
        n_keywords=len(kw_items),
    )

# Build per-article keyword map (duplicate keyword inside one article -> mean strength)
article_kw_strength = {}
for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])
    tmp = defaultdict(list)
    for kw_term, kw_strength in row["keyword_items"]:
        tmp[kw_term].append(float(kw_strength))
    article_kw_strength[article_id] = {k: sum(v) / len(v) for k, v in tmp.items()}

# Inverted indices
author_to_articles = defaultdict(set)
keyword_to_articles = defaultdict(list)

for _, row in articles.iterrows():
    article_id = row["article_id"] if row["article_id"] else normalize_id_from_pdf(row["pdf_file"])

    for author in row["authors"]:
        author_to_articles[author.lower()].add(article_id)

    for kw_term, kw_strength in article_kw_strength[article_id].items():
        keyword_to_articles[kw_term].append((article_id, kw_strength))

# Aggregate one edge per article pair from keywords
pair_acc = {}

def pair_key(a, b) -> tuple:
    """Return a canonical (sorted) edge key so (u,v) and (v,u) map to the same entry."""
    return (a, b) if a < b else (b, a)

for kw_term, art_strength_list in keyword_to_articles.items():
    if len(art_strength_list) < 2:
        continue

    for (u, su), (v, sv) in itertools.combinations(art_strength_list, 2):
        k = pair_key(u, v)
        if k not in pair_acc:
            pair_acc[k] = {
                "kw_scores": [],
                "kw_terms": set(),
                "shared_authors": set(),
                "shared_authors_count": 0,
            }
        pair_score = (float(su) + float(sv)) / 2.0
        pair_acc[k]["kw_scores"].append(pair_score)
        pair_acc[k]["kw_terms"].add(kw_term)

# Aggregate author overlaps; by default only enrich existing keyword edges
for author, arts in author_to_articles.items():
    arts = sorted(arts)
    if len(arts) < 2:
        continue
    for u, v in itertools.combinations(arts, 2):
        k = pair_key(u, v)
        if k not in pair_acc and not INCLUDE_AUTHOR_ONLY_EDGES:
            continue
        if k not in pair_acc:
            pair_acc[k] = {
                "kw_scores": [],
                "kw_terms": set(),
                "shared_authors": set(),
                "shared_authors_count": 0,
            }
        pair_acc[k]["shared_authors"].add(author)

# Build candidate edges + thresholds
candidate_edges = []
for (u, v), acc in pair_acc.items():
    kw_scores = acc["kw_scores"]
    kw_terms = sorted(acc["kw_terms"])
    kw_count = len(kw_terms)

    shared_authors = sorted(acc["shared_authors"])
    shared_authors_count = len(shared_authors)

    has_kw = kw_count > 0
    has_author = shared_authors_count > 0

    if has_kw:
        keyword_strength_mean = sum(kw_scores) / len(kw_scores)
        max_pair_score = max(kw_scores)
    else:
        keyword_strength_mean = -1.0
        max_pair_score = -1.0

    # Final score used for pruning/ranking
    final_score = keyword_strength_mean
    if has_kw and has_author:
        final_score += AUTHOR_BOOST * min(shared_authors_count, 2)

    # Edge filtering
    keep = False
    if has_kw:
        keep = (
            (kw_count >= MIN_SHARED_KEYWORDS)
            and (keyword_strength_mean >= MIN_EDGE_STRENGTH)
            and (max_pair_score >= MIN_MAX_PAIR_SCORE)
        )
    elif has_author and INCLUDE_AUTHOR_ONLY_EDGES:
        keep = True

    if not keep:
        continue

    edge_type = "author+keyword" if (has_kw and has_author) else ("keyword" if has_kw else "author")

    candidate_edges.append({
        "u": u,
        "v": v,
        "edge_type": edge_type,
        "strength": float(final_score),
        "keyword_strength_mean": float(keyword_strength_mean),
        "max_pair_score": float(max_pair_score),
        "shared_keywords_count": kw_count,
        "shared_keywords": "|".join(kw_terms),
        "shared_authors_count": shared_authors_count,
        "shared_authors": "|".join(shared_authors),
    })

# Top-K per node pruning
adj = defaultdict(list)
for idx, e in enumerate(candidate_edges):
    adj[e["u"]].append((idx, e["strength"]))
    adj[e["v"]].append((idx, e["strength"]))

top_ids_by_node = {}
for node, arr in adj.items():
    arr_sorted = sorted(arr, key=lambda x: x[1], reverse=True)
    top_ids_by_node[node] = {idx for idx, _ in arr_sorted[:TOP_K_PER_NODE]}

kept_edge_ids = set()
for idx, e in enumerate(candidate_edges):
    u, v = e["u"], e["v"]
    in_u = idx in top_ids_by_node.get(u, set())
    in_v = idx in top_ids_by_node.get(v, set())

    if USE_MUTUAL_KNN:
        if in_u and in_v:
            kept_edge_ids.add(idx)
    else:
        if in_u or in_v:
            kept_edge_ids.add(idx)

# Global edge cap after top-k pruning (keeps strongest edges only)
kept_edges = [candidate_edges[idx] for idx in kept_edge_ids]
kept_edges = sorted(kept_edges, key=lambda x: x["strength"], reverse=True)
if MAX_TOTAL_EDGES is not None:
    kept_edges = kept_edges[:MAX_TOTAL_EDGES]

# Add final edges to graph
for e in kept_edges:
    G.add_edge(
        e["u"],
        e["v"],
        edge_type=e["edge_type"],
        strength=e["strength"],
        keyword_strength_mean=e["keyword_strength_mean"],
        shared_keywords_count=e["shared_keywords_count"],
        shared_keywords=e["shared_keywords"],
        shared_authors_count=e["shared_authors_count"],
        shared_authors=e["shared_authors"],
    )

print("nodes:", G.number_of_nodes())
print("candidate_edges (pre top-k):", len(candidate_edges))
print("edges (post top-k, pre-cap):", len(kept_edge_ids))
print("edges (final, post-cap):", G.number_of_edges())

nodes: 2376
candidate_edges (pre top-k): 2366
edges (post top-k, pre-cap): 1819
edges (final, post-cap): 1819


**Export to GraphML.** Persists the pruned graph so it can be loaded by downstream notebooks without rebuilding it.

In [ ]:
nx.write_graphml(G, GRAPHML_OUT)
print("GraphML written to:", GRAPHML_OUT)

**Interactive visualisation.** Loads the GraphML and renders an interactive Pyvis HTML network — node size reflects keyword count, edge colour reflects edge type.

In [ ]:
G_loaded = nx.read_graphml(GRAPHML_OUT)

net = Network(height="900px", width="100%", bgcolor="#111111", font_color="white", notebook=False)

# Start with physics disabled to avoid UI freezes on larger graphs.
net.toggle_physics(False)

for n, nd in G_loaded.nodes(data=True):
    title = nd.get("title", n)
    net.add_node(
        n,
        label=title[:70],
        title=f"{title}<br>Authors: {nd.get('authors','')}",
        value=max(1, int(float(nd.get("n_keywords", 1)))),
    )

for u, v, ed in G_loaded.edges(data=True):
    edge_type = ed.get("edge_type", "")
    strength = float(ed.get("strength", -1))
    kw_count = int(float(ed.get("shared_keywords_count", 0)))
    au_count = int(float(ed.get("shared_authors_count", 0)))

    if edge_type == "author":
        color = "#7f8c8d"
        width = 1.0 + 0.2 * au_count
        title = f"type=author | shared_authors_count={au_count}"
    elif edge_type == "keyword":
        color = "#00c2ff"
        width = 1.0 + 0.4 * max(1.0, strength - 2.0)
        title = f"type=keyword | shared_keywords_count={kw_count} | score={strength:.3f}"
    else:
        color = "#ff9f43"
        width = 1.2 + 0.4 * max(1.0, strength - 2.0)
        title = (
            f"type=author+keyword | shared_authors_count={au_count} "
            f"| shared_keywords_count={kw_count} | score={strength:.3f}"
        )

    net.add_edge(u, v, color=color, width=width, title=title)

net.write_html(HTML_OUT, open_browser=False)
print("HTML written to:", HTML_OUT)
print("Open the HTML file in your browser (do not embed via IFrame in notebook).")

**Graph diagnostics.** Prints node/edge counts, edge-type breakdown, keyword-strength statistics, and connected-component sizes.

In [16]:
edge_type_counts = defaultdict(int)
for _, _, d in G.edges(data=True):
    edge_type_counts[d.get("edge_type", "unknown")] += 1

print("nodes:", G.number_of_nodes())
print("edges:", G.number_of_edges())
print("edge type counts:", dict(edge_type_counts))

if G.number_of_edges() > 0:
    strengths = [float(d.get("strength", 0.0)) for _, _, d in G.edges(data=True) if d.get("edge_type") != "author"]
    if strengths:
        print("keyword edge strength min/mean/max:", min(strengths), sum(strengths) / len(strengths), max(strengths))

components = sorted(nx.connected_components(G), key=len, reverse=True)
print("connected components:", len(components))
print("largest component size:", len(components[0]) if components else 0)

nodes: 2376
edges: 1819
edge type counts: {'author+keyword': 431, 'keyword': 1388}
keyword edge strength min/mean/max: 3.2 3.5856529146430978 5.007936507936509
connected components: 1302
largest component size: 829
